# Supplementary Material S1 — reproduction on Google Colab

*Operating-point shift and the frozen stability margin after resistive load shedding* (O. Ayvaz, Ö. Tomak)

This notebook runs every reproduction script of Supplementary Material S1 and checks its output against the distributed files. The scripts and the sealed records are not changed. The notebook runs them on a separate working copy:

1. It checks the SHA-256 manifests of the archive. Nothing is run if a sealed file differs.
2. It creates an isolated Python 3.12 environment with the pinned versions in `requirements_pinned.txt`.
3. It runs the ten steps in the documented order: the hash check of the preregistered run, Tables 7–8, the post-hoc statistics, the local and abrupt nonlinear checks, the revision cross-checks and Figures 1–8.
4. It compares every regenerated CSV/JSON file with the distributed copy, shows the report and the figures, and offers a zip of the results for download.

**How to use:** keep the default CPU runtime (no GPU is needed) and choose *Runtime ▸ Run all*. When asked, upload `Supplementary_S1_data_code_v1.1.zip`. You can also read the archive from Google Drive or from a URL; see the settings cell. Expected time: about 1–2 min for the environment and about 2–3 min for the runs.

The notebook also accepts the GitHub mirror of the archive. The preregistration files are not part of the mirror, so the hash check of step S0 is reported as *not applicable* there.

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------------
ARCHIVE_SOURCE = "upload"   # "upload" | "drive" | "url" | "local"
DRIVE_PATH     = "/content/drive/MyDrive/Supplementary_S1_data_code_v1.1.zip"
ARCHIVE_URL    = ""         # direct link to the zip (e.g. a GitHub release asset or a Zenodo file)
LOCAL_PATH     = "Supplementary_S1_data_code_v1.1.zip"
USE_PINNED_ENV = True       # True: isolated Python 3.12 + requirements_pinned.txt (recommended)
                            # False: use the packages already installed in this runtime

In [ ]:
# ---- 1. Get and extract the archive --------------------------------------------------------------------
import os, sys, shutil, subprocess, zipfile, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if IN_COLAB else Path.cwd()

if ARCHIVE_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()                      # saved in the current folder
    archive = Path(next(iter(uploaded))).resolve()
elif ARCHIVE_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    archive = Path(DRIVE_PATH)
elif ARCHIVE_SOURCE == "url":
    archive = BASE / "S1_archive.zip"
    urllib.request.urlretrieve(ARCHIVE_URL, archive)
else:
    archive = Path(LOCAL_PATH).resolve()

extract_to = BASE / "S1"
if extract_to.exists():
    shutil.rmtree(extract_to)
with zipfile.ZipFile(archive) as z:
    z.extractall(extract_to)
ROOT = min((p.parent for p in extract_to.rglob("requirements_pinned.txt")), key=lambda p: len(p.parts))
print("Archive:", archive.name, "->", ROOT)

In [ ]:
# ---- 2. Python environment -----------------------------------------------------------------------------
def run_live(cmd, check=True):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed with exit code %d" % rc)
    return rc

PY = sys.executable
if USE_PINNED_ENV:
    try:
        venv = BASE / "segan_env"
        run_live([sys.executable, "-m", "pip", "install", "-q", "uv"])
        run_live([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--clear", venv])
        PY = venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
        run_live([sys.executable, "-m", "uv", "pip", "install", "--python", PY, "-r", ROOT / "requirements_pinned.txt"])
    except Exception as exc:
        print("\nThe pinned environment could not be created (%s).\nFalling back to the packages of this runtime; "
              "small numerical differences are possible and are reported." % exc)
        PY = sys.executable
if str(PY) == sys.executable:
    run_live([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "sympy", "matplotlib", "pillow"])
PY = str(PY)
print("\nScripts will run with:", PY)

In [ ]:
# ---- 3. Run and check all reproduction scripts ---------------------------------------------------------
OUT = BASE / "segan_s1_results"
rc = run_live([sys.executable, "-u", ROOT / "colab" / "run_all.py", "--root", ROOT, "--python", PY, "--out", OUT],
              check=False)
print("\nexit code:", rc, "(0 = all steps reproduced)")

In [ ]:
# ---- 4. Report and figures -----------------------------------------------------------------------------
from IPython.display import Markdown, Image, display
display(Markdown((OUT / "REPORT.md").read_text(encoding="utf-8")))
fig_dir = OUT / "regenerated" / "posthoc_revision_20261025" / "figures"
for n in range(1, 9):
    png = fig_dir / f"Figure_{n}.png"
    if png.exists():
        display(Markdown(f"**Figure {n}** (regenerated)"))
        display(Image(filename=str(png), width=760))

In [ ]:
# ---- 5. Download the results ---------------------------------------------------------------------------
zip_path = shutil.make_archive(str(BASE / "segan_s1_results"), "zip", OUT)
print("Results:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

### Reading the result

- **ALL REPRODUCED** means three things: the archive was intact, every script ran, and every regenerated number agrees with the distributed file within the stated tolerance. The tolerance absorbs round-off and solver differences between machines; it is not wide enough to hide a change of result. Figures are checked for presence, and their content should be compared by eye with the manuscript.
- The 79-member preregistered HDL2b run itself is **not** re-executed. Step S0 only verifies the hashes of its preregistration and of its 13 pinned inputs, as described in the manuscript. The sealed member records in `research_intelligence/cdl1/out/HDL2b_20261021_a/` are the inputs of all other steps.